In [2]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import timedelta
import math

# Initialize Faker with Australian locale
np.random.seed(42)
fake = Faker('en_AU')

NUM_ORDERS = 5000
NUM_CUSTOMERS = 500 # Typical B2B scenario: fewer customers, higher repurchase rate

# ==========================================
# 1. Define Warehouse Coordinates (Top 4 AU Locations)
# ==========================================
WAREHOUSES = {
    'Melbourne': {'lat': -37.8136, 'lon': 144.9631},
    'Brisbane':  {'lat': -27.4698, 'lon': 153.0251},
    'Adelaide':  {'lat': -34.9285, 'lon': 138.6007},
    'Perth':     {'lat': -31.9505, 'lon': 115.8605}
}

# Helper Function: Calculate spherical distance between two points
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0 # Earth radius in kilometers
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    c = 2 * math.asin(math.sqrt(a))
    return R * c

# ==========================================
# 2. Generate B2B Customer Pool (Entity-Level Data)
# ==========================================
print("Generating Australian B2B customer profiles and contract details...")
customers = {}

# Major Australian commercial hubs to simulate realistic customer distribution
au_hubs = [
    (-33.86, 151.20), # Sydney (High customer density, routed to MEL or BNE)
    (-37.81, 144.96), # Melbourne
    (-27.47, 153.02), # Brisbane
    (-31.95, 115.86), # Perth
    (-34.93, 138.60), # Adelaide
    (-32.92, 151.78), # Newcastle
]

for _ in range(NUM_CUSTOMERS):
    cust_id = f"B2B-{fake.unique.random_number(digits=5)}"
    
    # Generate actual customer address with random offset around commercial hubs
    base_lat, base_lon = au_hubs[np.random.randint(0, len(au_hubs))]
    cust_lat = base_lat + np.random.normal(0, 2)
    cust_lon = base_lon + np.random.normal(0, 2)
    
    # Find the nearest warehouse
    min_dist = float('inf')
    nearest_wh = None
    for wh_name, wh_coords in WAREHOUSES.items():
        dist = haversine(cust_lat, cust_lon, wh_coords['lat'], wh_coords['lon'])
        if dist < min_dist:
            min_dist = dist
            nearest_wh = wh_name
            
    # B2B Contract Terms: Randomly assign 3 or 7 days of Late Delivery grace period
    grace_period = np.random.choice([3, 7], p=[0.7, 0.3])
    
    customers[cust_id] = {
        'Customer_Address': fake.address().replace('\n', ', '),
        'Customer_Lat': cust_lat,
        'Customer_Lon': cust_lon,
        'Nearest_Warehouse': nearest_wh,
        'Distance_to_Warehouse_km': round(min_dist, 2),
        'Contract_Grace_Period_Days': grace_period
    }

# ==========================================
# 3. Generate Order Data (Transaction-Level Data)
# ==========================================
print("Generating order and fulfillment data...")
data = []
customer_ids = list(customers.keys())

for i in range(NUM_ORDERS):
    order_id = f"ORD-{100000 + i}"
    cust_id = np.random.choice(customer_ids)
    cust_info = customers[cust_id]
    
    order_date = fake.date_time_between(start_date='-1y', end_date='now')
    # B2B required delivery days are typically longer, assuming 7-14 days
    required_days = np.random.randint(7, 15)
    required_delivery_date = order_date + timedelta(days=required_days)
    
    # The further the distance, the higher the delay probability and variance
    dist_km = cust_info['Distance_to_Warehouse_km']
    delay_mean = dist_km / 1000.0  # +1 day average delay per 1000 km
    delay_std = 2.0 + (dist_km / 800.0)
    
    delay_days = int(np.random.normal(loc=delay_mean, scale=delay_std))
    actual_delivery_date = required_delivery_date + timedelta(days=delay_days)
    
    # Base shipping cost + distance-based pricing
    shipping_cost = round(50.0 + (dist_km * 0.05) + np.random.normal(0, 10), 2)
    shipping_cost = max(20.0, shipping_cost)
    
    data.append({
        "Order_ID": order_id,
        "Customer_ID": cust_id,
        "Order_Date": order_date,
        "Required_Delivery_Date": required_delivery_date,
        "Actual_Delivery_Date": actual_delivery_date,
        "Delay_Days": delay_days,
        "Shipping_Cost": shipping_cost,
        "Contract_Grace_Period_Days": cust_info['Contract_Grace_Period_Days'],
        "Warehouse_Location": cust_info['Nearest_Warehouse'],
        "Distance_to_Warehouse_km": cust_info['Distance_to_Warehouse_km']
    })

df = pd.DataFrame(data)

# ==========================================
# 4. B2B Tiered Complaint Probability Model
# ==========================================
def determine_complaint(row):
    delay = row['Delay_Days']
    grace_period = row['Contract_Grace_Period_Days']
    
    # 1. Delivered early or on time (<= 0 days)
    if delay <= 0:
        # Extremely low probability of logistics complaint, assumed 1%
        prob = 0.01
    
    # 2. Gray Area: Delayed, but within the contract grace period (0 < delay <= grace_period)
    elif delay <= grace_period:
        # Customers cannot claim breach, but will still be dissatisfied. 
        # Probability increases as it approaches the grace limit (e.g., 10% - 35%)
        base_gray_prob = 0.10
        increment = (0.25 / grace_period) * delay
        prob = base_gray_prob + increment
        
    # 3. Complete Breach: Exceeded the contract grace period (delay > grace_period)
    else:
        # B2B breach complaint rate is highly penalized: base 70%, up to 99%
        overdue_days = delay - grace_period
        prob = min(0.70 + (0.05 * overdue_days), 0.99)
        
    # Generate final binary label using random threshold
    is_complained = 1 if np.random.random() < prob else 0
    
    # Assign specific complaint reason
    if is_complained == 0:
        reason = None
    else:
        if delay <= 0:
             reason = np.random.choice(['Quality Issue', 'Packaging Damage'], p=[0.7, 0.3])
        elif delay <= grace_period:
             # In the gray area, clients often use "Customer Service" or "Late Delivery" as excuses
             reason = np.random.choice(['Late Delivery', 'Customer Service'], p=[0.8, 0.2])
        else:
             # A definitive breach always results in a Late Delivery complaint
             reason = 'Late Delivery' 
             
    return pd.Series([is_complained, reason])

print("Calculating B2B complaint probability logic...")
df[['Is_Complained', 'Complaint_Reason']] = df.apply(determine_complaint, axis=1)

# Save the generated data to a CSV file
file_path = 'b2b_synthetic_delivery_data.csv'
df.to_csv(file_path, index=False)
print(f"✅ Generation complete! File saved to: {file_path}")

# ==========================================
# 5. Business Logic Validation
# ==========================================
print("\n--- Warehouse Allocation Statistics ---")
print(df['Warehouse_Location'].value_counts())

print("\n--- Gray Area (Within Grace Period) Complaint Validation ---")
gray_area_df = df[(df['Delay_Days'] > 0) & (df['Delay_Days'] <= df['Contract_Grace_Period_Days'])]
if len(gray_area_df) > 0:
    gray_rate = gray_area_df['Is_Complained'].mean()
    print(f"Total compliant but delayed orders: {len(gray_area_df)}")
    print(f"Gray area complaint rate: {gray_rate:.2%} (Validates the reality of receiving complaints within the legal 3/7-day buffer)")

Generating Australian B2B customer profiles and contract details...
Generating order and fulfillment data...
Calculating B2B complaint probability logic...
✅ Generation complete! File saved to: b2b_synthetic_delivery_data.csv

--- Warehouse Allocation Statistics ---
Warehouse_Location
Brisbane     1786
Melbourne    1562
Perth         833
Adelaide      819
Name: count, dtype: int64

--- Gray Area (Within Grace Period) Complaint Validation ---
Total compliant but delayed orders: 1690
Gray area complaint rate: 21.07% (Validates the reality of receiving complaints within the legal 3/7-day buffer)
